In [3]:
# Question 1: RSA Encryption

def ConvertToInt(message_str):
  res = 0
  for i in range(len(message_str)):
    res = res * 256 + ord(message_str[i])
  return res

def PowMod(a, n, mod):
    if n == 0:
        return 1 % mod
    elif n == 1:
        return a % mod
    else:
        b = PowMod(a, n // 2, mod)
        b = b * b % mod
        if n % 2 == 0:
          return b
        else:
          return b * a % mod

def Encrypt(message, modulo, exponent):
  message_int = ConvertToInt(message)
  return PowMod(message_int, exponent, modulo)

In [5]:
# Question 2: RSA Decryption

def ConvertToStr(n):
    res = ""
    while n > 0:
        res += chr(n % 256)
        n //= 256
    return res[::-1]

def ExtendedEuclid(a, b):
    if b == 0:
        return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0:
        b = (b % n + n) % n
    return b

def Decrypt(ciphertext, p, q, exponent):
  modulo = p * q
  phi = (p - 1) * (q - 1)
  d = InvertModulo(exponent, phi)
  plaintext = PowMod(ciphertext, d, modulo)
  return ConvertToStr(plaintext)

a = 3
b = 7
c = InvertModulo(a, b)
print(c)

p = 1000000007
q = 1000000009
exponent = 23917
modulo = p * q
ciphertext = Encrypt("attack", modulo, exponent)
message = Decrypt(ciphertext, p, q, exponent)
print(message)

5
attack


In [6]:
# Question 3: Decipher Simple RSA Message

def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
  for message in potential_messages:
    encrypted_message = Encrypt(message, modulo, exponent)

    if encrypted_message == ciphertext:
      return message

  return "don't know"

modulo = 101
exponent = 12
ciphertext = Encrypt("attack", modulo, exponent)
print(ciphertext)
print(DecipherSimple(ciphertext, modulo, exponent, ["attack", "don't attack", "wait"]))

78
attack


In [7]:
# Question 4: Decipher RSA Using Small Prime Factor

def DecipherSmallPrime(ciphertext, modulo, exponent):
  for p in range(2, 1000000):
    if modulo % p == 0:
      q = modulo // p
      return Decrypt(ciphertext, p, q, exponent)

  return "don't know"

modulo = 101 * 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387

exponent = 239
ciphertext = Encrypt("attack", modulo, exponent)
print(ciphertext)
print(DecipherSmallPrime(ciphertext, modulo, exponent))

1773488865857950936127070415107286811735036786757360876488901668828411065415319121739305112809023074111043494948978081751851444797671426477224221428915534085499185651993502038149646686545099252921036161028865989784470806731700777521041215564203613423323285066203840329997825032261425978638927438518035030191944107572667758243311046550229409822846149831396410991755012812007339251536452615590635223780672174679677196279304461931703846255925914845500897273821827255506913276266523704922524156943959725369352703672131222283682166791913172292046709244906816668026610802197814114770298014914576347693504969280551524366327904


RecursionError: maximum recursion depth exceeded

In [8]:
# Question 5: Decipher RSA Using Small Difference Between Prime Factors

import math

def IntSqrt(n):
    return math.isqrt(n)

def DecipherSmallDiff(ciphertext, modulo, exponent):
  sqrtn = IntSqrt(modulo)

  x = sqrtn
  if x * x < modulo:
    x += 1

  while True:
    y_squared = x * x - modulo
    y = IntSqrt(y_squared)

    if y * y == y_squared:
      p = x - y
      q = x + y

      if p * q == modulo:
        return Decrypt(ciphertext, p, q, exponent)

    x += 1

p = 1000000007
q = 1000000009
n = p * q
e = 239
ciphertext = Encrypt("attack", n, e)
message = DecipherSmallDiff(ciphertext, n, e)
print(ciphertext)
print(message)

48582223151298950
attack


In [9]:
# Question 6: RSA Common Divisor Attack

def GCD(a, b):
  if b == 0:
    return a
  return GCD(b, a % b)

def DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent, second_ciphertext, second_modulo, second_exponent):

  common_prime = GCD(first_modulo, second_modulo)

  if common_prime == 1:
    return "don't know"

  q1 = first_modulo // common_prime
  q2 = second_modulo // common_prime

  first_message = Decrypt(first_ciphertext, common_prime, q1, first_exponent)
  second_message = Decrypt(second_ciphertext, common_prime, q2, second_exponent)

  return (first_message, second_message)

# Example usage with common prime p and different second primes q1 and q2

p = 101
q1 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387

q2 = 1000000007
first_modulo = p * q1
second_modulo = p * q2
first_exponent = 239
second_exponent = 17
first_ciphertext = Encrypt("attack", first_modulo, first_exponent)
second_ciphertext = Encrypt("wait", second_modulo, second_exponent)

print(DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent, second_ciphertext, second_modulo, second_exponent))

RecursionError: maximum recursion depth exceeded

In [10]:
# Question 7: Hastad's Broadcast Attack

def DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo):

  inverse = InvertModulo(first_modulo, second_modulo)

  k = ((second_ciphertext - first_ciphertext) * inverse) % second_modulo

  m_squared = first_ciphertext + first_modulo * k

  m = IntSqrt(m_squared)

  return ConvertToStr(m)


p1 = 790383132652258876190399065097
q1 = 662503581792812531719955475509
p2 = 656917682542437675078478868539
q2 = 1263581691331332127259083713503

n1 = p1 * q1
n2 = p2 * q2

ciphertext1 = Encrypt("attack", n1, 2)
ciphertext2 = Encrypt("attack", n2, 2)

message = DecipherHastad(ciphertext1, n1, ciphertext2, n2)
print(message)

attack
